# 01. Exploration of Raw Data.

I've retreived the blocks and transactions using Google Big Query to used for training data to build our Unsupervised cluster in order to inspect what type of patterns arise, and get some information about the data. 

* 1 year worth of blocks (and its transactions), and transfer event logs were pulled from BigQuery's public dataset then fetched into local storage in paraquet format for downstream features.


* First with authenticate with `gcloud auth application-default login`


## Raw Block entries (Big Query vs. Alchemy via RPC)

Requires authenticating using `gcloud auth application-default login` first. I am generating two dataframes to demonstrate side by side.

In [108]:
# import relevant libraries and set up BigQuery client and RPC function
from google.cloud import bigquery
client = bigquery.Client(project="chainsense-497110")

import requests, os, dotenv, pandas as pd
dotenv.load_dotenv()
def rpc(method, params):
    url = f"https://eth-mainnet.g.alchemy.com/v2/{os.getenv("ALCHEMY_KEY")}"
    r = requests.post(url, json={"jsonrpc": "2.0", "id":1, "method":method, "params":params})
    return r.json()["result"]

In [113]:
sample = client.query("""
    SELECT * FROM `bigquery-public-data.crypto_ethereum.blocks`
    ORDER BY number DESC
    LIMIT 1
""").to_dataframe()
bq_df = (
    sample.T # Take 1 row sample (column major) and convert into row major
    .reset_index() # after transpose, the original column names are now the index of the dataframe
    .rename(columns={"index": "field", 0: "value"}) # 0 because after transpose the single row had label 0
    .assign(type=lambda df: df["value"].apply(lambda v: type(v).__name__))
    [["field", "type", "value"]] # reorder columns to be more intuitive
)
block = rpc("eth_getBlockByNumber", ["latest", True])
rpc_df = pd.DataFrame([(k,type(v).__name__,v) for k,v in block.items()], columns=["field", "type", "value"])

/Users/mr.po/Documents/GitHub/ucsd-ChainSense/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
# Big Query Blocks vs RPC (eth_getBlockByNumber) Blocks
pd.concat([bq_df, rpc_df], axis=1)

,field,type,value,field,type,value
0,timestamp,Timestamp,2026-05-22 23:59:59+00:00,hash,str,0x09024c13961b37ce98856b716cd7e309ef26fe637025...
1,number,int,25154243,parentHash,str,0x42984a35000eaaab94a63c31cab448ec0a3383275169...
2,hash,str,0x26016d681996792f2d2d39863b7f100b1b937b4c5e37...,sha3Uncles,str,0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a...
3,parent_hash,str,0x901f793f099117eb211cf5977a1af393dd4e13469191...,miner,str,0xdadb0d80178819f2319190d340ce9a924f783711
4,nonce,str,0x0000000000000000,stateRoot,str,0xbbab1571b37f18a95d86107fd601683bd8207635f9ab...
5,sha3_uncles,str,0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a...,transactionsRoot,str,0xf3fc685c16050f817d35b460972d16d66a8b0f93bcfa...
6,logs_bloom,str,0xbdab57c5f7d06dd87ffd7c3ee2eb96dfdfd5fc4eb677...,receiptsRoot,str,0x75740c9c2b6b7eab2f0e66cf2895e55224cc9fe23112...
7,transactions_root,str,0x01f1ade969acfebb8e0d94b88e6618e2fc7fcce3cb91...,logsBloom,str,0x87f173cb67fbfffd71d77fffbccff45b732da8fb048d...
8,state_root,str,0x3c2e0634907eb91659d9d70c11d38ac6a3ad5f0d06af...,difficulty,str,0x0
9,receipts_root,str,0xe9ff53d1e2f06495acf0bdf93b218dac7a5e1877bfbf...,number,str,0x17fde66


## Raw Tx entries (Big Query vs. Alchemy via RPC)

In [ ]:
bq_tx = client.query("""
    SELECT * FROM `bigquery-public-data.crypto_ethereum.transactions`
    WHERE DATE(block_timestamp) = CURRENT_DATE() - 1
    LIMIT 1
""").to_dataframe()
bq_tx = bq_tx.T.reset_index().rename(columns={"index": "field", 0: "value"}).assign(type=lambda df: df["value"].apply(lambda v: type(v).__name__))[["field", "type", "value"]]
rpc_tx = block["transactions"][0]
rpc_tx = pd.DataFrame([(k,type(v).__name__,v) for k,v in rpc_tx.items()], columns=["field", "type", "value"])

/Users/mr.po/Documents/GitHub/ucsd-ChainSense/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [121]:
pd.concat([bq_tx, rpc_tx], axis=1)

,field,type,value,field,type,value
0,hash,str,0x0d259a89d7d22858d988a1ccb65f73a55ea31882fdac...,type,str,0x2
1,nonce,int,50073,chainId,str,0x1
2,transaction_index,int,102,nonce,str,0x8c27
3,from_address,str,0x6402076597364f810bb553b85d67d597df92d823,gas,str,0x88b8
4,to_address,str,0x45118b56d5066a6352d78d15f795cfab33c4def3,maxFeePerGas,str,0x27f9b8f4
5,value,Decimal,0E-9,maxPriorityFeePerGas,str,0x1cfadc84
6,gas,int,36547,to,str,0x681e908b8ab57c49c74d770f369754ccc3e1ae09
7,gas_price,int,195370959,value,str,0x0
8,input,str,0xe4715bb5000000000000000000000000000000000000...,accessList,list,[]
9,receipt_cumulative_gas_used,int,5775443,input,str,0x6a117b3186c930b461005de552725c0005f490e30700...


## Raw Log entries (Big Query vs. Alchemy via RPC)

In [ ]:
# Cell 9: Address landscape
# - codes.is_contract.mean()
# - Distribution of code_size for contracts (small = proxy/clones, large = full contracts)
# - Cross-check: of the 'to' addresses in contract calls, what fraction are classified as contracts? (should be ~100%)

In [ ]:
# Cell 10: First takeaways (markdown cell)
# Write 3-5 bullet points: what surprised you, what looks healthy, what to investigate.
# This is the cell that proves you actually looked at the data.